In [5]:
import os
from utils import *
from os.path import join
from pprint import pprint

import numpy as np
from scipy.stats import circmean
from math import radians

import requests
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
import os
import json
import model_helpers as mh
from neuron import h, load_mechanisms

In [10]:
code_version = 'Hay'
model_version = 'NeuroML'
nmldb_id = 'NMLCL000073'  # 'NMLCL000073' (Hay et al. 2011)
model_name = f'{nmldb_id}-{model_version}'
mh.download_from_nmldb(nmldb_id, model_version)

Model NMLCL000073 already downloaded.


In [11]:
sim_name = 'morphology'
cwd = os.getcwd()
models_dir = os.path.join(cwd, 'models') 
model_dir = os.path.join(models_dir, model_version, model_name)  # 'L5bPCmodelsEH')
hocs_dir = model_dir if 'biophys' not in model_name else os.path.join(model_dir,'models')
mod_dir = model_dir if 'biophys' not in model_name else os.path.join(model_dir, 'mod')
output_dir, sim_dir = mh.create_output_dirs(sim_name, model_dir)
output_dir, sim_dir

('/home/kedoxey/CRCNS/PyramidalCellSimulations/models/NeuroML/NMLCL000073-NeuroML/output',
 '/home/kedoxey/CRCNS/PyramidalCellSimulations/models/NeuroML/NMLCL000073-NeuroML/output/morphology')

In [14]:
cell_name = 'L5PC'
nml_file = os.path.join(model_dir,cell_name+'.cell.nml')
model_tree = ET.parse(nml_file)

In [15]:
root = model_tree.getroot()

In [16]:
root.tag, root.attrib

('{http://www.neuroml.org/schema/neuroml2}neuroml',
 {'{http://www.w3.org/2001/XMLSchema-instance}schemaLocation': 'http://www.neuroml.org/schema/neuroml2  https://raw.githubusercontent.com/NeuroML/NeuroML2/development/Schemas/NeuroML2/NeuroML_v2beta4.xsd',
  'id': 'L5PC'})

In [17]:
for child in root:
    print(child.tag, child.attrib)

{http://www.neuroml.org/schema/neuroml2}include {'href': 'Ca_HVA.channel.nml'}
{http://www.neuroml.org/schema/neuroml2}include {'href': 'Ca_LVAst.channel.nml'}
{http://www.neuroml.org/schema/neuroml2}include {'href': 'CaDynamics_E2_NML2__decay122__gamma5_09Emin4.nml'}
{http://www.neuroml.org/schema/neuroml2}include {'href': 'CaDynamics_E2_NML2__decay460__gamma5_01Emin4.nml'}
{http://www.neuroml.org/schema/neuroml2}include {'href': 'Ih.channel.nml'}
{http://www.neuroml.org/schema/neuroml2}include {'href': 'Im.channel.nml'}
{http://www.neuroml.org/schema/neuroml2}include {'href': 'K_Pst.channel.nml'}
{http://www.neuroml.org/schema/neuroml2}include {'href': 'K_Tst.channel.nml'}
{http://www.neuroml.org/schema/neuroml2}include {'href': 'Nap_Et2.channel.nml'}
{http://www.neuroml.org/schema/neuroml2}include {'href': 'NaTa_t.channel.nml'}
{http://www.neuroml.org/schema/neuroml2}include {'href': 'pas.channel.nml'}
{http://www.neuroml.org/schema/neuroml2}include {'href': 'SK_E2.channel.nml'}
{ht

In [18]:
cell= root[-1]
cell

<Element '{http://www.neuroml.org/schema/neuroml2}cell' at 0x7f5c1e00f3b0>

In [19]:
for child in cell:
    print(child.tag, child.attrib)

{http://www.neuroml.org/schema/neuroml2}notes {}
{http://www.neuroml.org/schema/neuroml2}morphology {'id': 'morphology_L5PC'}
{http://www.neuroml.org/schema/neuroml2}biophysicalProperties {'id': 'biophys'}


In [21]:
morpho = cell[1]
morpho

<Element '{http://www.neuroml.org/schema/neuroml2}morphology' at 0x7f5c1e00f4f0>

In [22]:
segmentGroups = {}
for i, group in enumerate(morpho.findall('{http://www.neuroml.org/schema/neuroml2}segmentGroup')):
    
    segmentGroups.update({i:group})


In [25]:
segmentGroups

{0: <Element '{http://www.neuroml.org/schema/neuroml2}segmentGroup' at 0x7f5c1d6fae00>,
 1: <Element '{http://www.neuroml.org/schema/neuroml2}segmentGroup' at 0x7f5c1d6ffb80>,
 2: <Element '{http://www.neuroml.org/schema/neuroml2}segmentGroup' at 0x7f5c1d6ffd10>,
 3: <Element '{http://www.neuroml.org/schema/neuroml2}segmentGroup' at 0x7f5c1d704590>,
 4: <Element '{http://www.neuroml.org/schema/neuroml2}segmentGroup' at 0x7f5c1d704900>,
 5: <Element '{http://www.neuroml.org/schema/neuroml2}segmentGroup' at 0x7f5c1d704f90>,
 6: <Element '{http://www.neuroml.org/schema/neuroml2}segmentGroup' at 0x7f5c1d708400>,
 7: <Element '{http://www.neuroml.org/schema/neuroml2}segmentGroup' at 0x7f5c1d7086d0>,
 8: <Element '{http://www.neuroml.org/schema/neuroml2}segmentGroup' at 0x7f5c1d70eb80>,
 9: <Element '{http://www.neuroml.org/schema/neuroml2}segmentGroup' at 0x7f5c1d70edb0>,
 10: <Element '{http://www.neuroml.org/schema/neuroml2}segmentGroup' at 0x7f5c1d692360>,
 11: <Element '{http://www.neur

In [36]:
comp_domains = {}
for key, val in segmentGroups.items():
    if val.attrib['id'] in ['soma_group', 'axon_group', 'apical_dends', 'basal_dends']:
        comp_domains.update({val.attrib['id']:val})

In [37]:
comp_domains

{'soma_group': <Element '{http://www.neuroml.org/schema/neuroml2}segmentGroup' at 0x7f5c1d3aa7c0>,
 'axon_group': <Element '{http://www.neuroml.org/schema/neuroml2}segmentGroup' at 0x7f5c1d367590>,
 'apical_dends': <Element '{http://www.neuroml.org/schema/neuroml2}segmentGroup' at 0x7f5c1d385310>,
 'basal_dends': <Element '{http://www.neuroml.org/schema/neuroml2}segmentGroup' at 0x7f5c1d327130>}

## Somatic compartments

In [39]:
comp_domains['soma_group']

<Element '{http://www.neuroml.org/schema/neuroml2}segmentGroup' at 0x7f5c1d3aa7c0>

In [40]:
for child in comp_domains['soma_group']:
    print(child.tag, child.attrib)

{http://www.neuroml.org/schema/neuroml2}include {'segmentGroup': 'soma_0'}


In [41]:
soma_group = comp_domains['soma_group'][0].attrib['segmentGroup']

In [42]:
soma_segments = {}
for segment in morpho.findall('{http://www.neuroml.org/schema/neuroml2}segment'):
    if soma_group in segment.attrib['name']:
        soma_segments.update({segment.attrib['id']:segment.attrib['name']})
soma_segments

{'0': 'Seg0_soma_0',
 '2': 'Seg1_soma_0',
 '3': 'Seg2_soma_0',
 '4': 'Seg3_soma_0',
 '5': 'Seg4_soma_0',
 '6': 'Seg5_soma_0',
 '7': 'Seg6_soma_0',
 '8': 'Seg7_soma_0',
 '9': 'Seg8_soma_0',
 '10': 'Seg9_soma_0',
 '11': 'Seg10_soma_0',
 '12': 'Seg11_soma_0',
 '13': 'Seg12_soma_0',
 '14': 'Seg13_soma_0',
 '15': 'Seg14_soma_0',
 '16': 'Seg15_soma_0',
 '17': 'Seg16_soma_0',
 '18': 'Seg17_soma_0',
 '19': 'Seg18_soma_0',
 '20': 'Seg19_soma_0'}

In [43]:
for child in morpho[0]:
    print(child.tag,child.attrib)

{http://www.neuroml.org/schema/neuroml2}proximal {'x': '34.1634', 'y': '17.6215', 'z': '-50.25', 'diameter': '3.80039'}
{http://www.neuroml.org/schema/neuroml2}distal {'x': '35.3196', 'y': '17.6937', 'z': '-50.25', 'diameter': '6.44373'}
